## Análise inicial

In [0]:
%sql
-- Verificando como o Spark interpretou o JSON ao criar tabela
-- ==============================================================================
-- DESCRIBE lh_nautical.lh_nautical_db.custos_importacao;

-- análise
-- ==============================================================================
SELECT * FROM lh_nautical.lh_nautical_db.custos_importacao;

category,historic_data,product_id,product_name
eletrônicos,"List(List(10/08/2016, 10583.63), List(15/06/2018, 8778.36), List(25/09/2018, 8023.87), List(19/03/2019, 8772.78), List(17/01/2020, 7918.18), List(17/06/2020, 6310.01), List(02/07/2021, 6586.7), List(16/05/2022, 6538.2), List(28/02/2023, 6360.91), List(17/10/2023, 6574.8), List(16/02/2024, 6657.12), List(22/02/2024, 6703.2), List(15/03/2024, 6633.66), List(02/08/2024, 5774.5), List(08/04/2025, 5579.75))",1,Transponder AIS Maré Magnum
eletrônicos,"List(List(23/11/2017, 4325.09), List(06/07/2022, 2577.22), List(24/05/2023, 2829.74), List(05/11/2025, 2602.27))",2,Transponder Furuno Marlin
eletrônicos,"List(List(12/04/2016, 2549.21), List(26/07/2018, 2423.45), List(27/06/2019, 2335.69), List(17/07/2019, 2399.28), List(14/10/2019, 2187.31), List(04/12/2020, 1745.49), List(21/06/2022, 1753.77))",3,Radar Furuno Pulse Leviathan
eletrônicos,"List(List(04/03/2016, 909.55), List(22/11/2016, 1010.42), List(19/04/2017, 1080.89), List(22/11/2018, 887.7), List(21/09/2022, 654.31), List(28/11/2023, 692.14), List(13/03/2024, 679.13), List(10/03/2025, 583.85))",4,Rádio AIS Hydro Tidal Zen
eletrônicos,"List(List(10/02/2016, 6006.45), List(17/01/2017, 7374.9), List(18/02/2021, 4364.4), List(03/05/2022, 4718.61), List(24/05/2022, 4920.79), List(15/08/2023, 4752.24), List(24/09/2024, 4327.37), List(24/07/2025, 4285.3))",5,Piloto Automático Furuno Storm
eletrônicos,"List(List(23/07/2020, 2288.92), List(01/12/2021, 2104.66), List(07/03/2024, 2394.79), List(21/06/2024, 2172.43), List(04/12/2024, 1951.33), List(19/03/2025, 2086.5))",6,Transponder AIS Vector
eletrônicos,"List(List(26/10/2016, 6253.41), List(21/06/2017, 5871.9), List(08/05/2018, 5454.91), List(12/12/2022, 3678.62), List(18/08/2025, 3604.64))",7,Radar AIS Zen
eletrônicos,"List(List(06/12/2019, 1193.04), List(19/06/2020, 932.31), List(18/02/2021, 919.04), List(06/08/2021, 951.1), List(23/03/2023, 947.09), List(10/05/2023, 1006.07), List(16/04/2024, 947.05), List(03/06/2025, 879.22))",8,GPS AIS Zen
eletrônicos,"List(List(26/12/2018, 10115.54), List(21/05/2020, 7088.62), List(10/12/2020, 7808.97), List(05/05/2022, 7933.96), List(16/04/2024, 7544.56), List(21/08/2024, 7258.64))",9,Transponder AIS Titan Pulse
eletrônicos,"List(List(29/05/2018, 8591.86), List(16/11/2021, 5849.07), List(20/12/2021, 5615.0), List(23/05/2023, 6449.43), List(29/06/2023, 6594.15), List(05/09/2023, 6445.41), List(13/11/2023, 6505.49), List(24/11/2023, 6547.91), List(07/10/2024, 5864.71), List(21/03/2025, 5596.76), List(01/10/2025, 6021.02))",10,Piloto Automático Simrad Titan Flux Magnum


## Padronização da tabela

In [0]:
%sql
-- Limpar e estruturar a tabela de Custos (Camada Prata).
-- Uso do EXPLODE para transformar o array de histórico em linhas individuais.
-- Padronização da categoria para facilitar análises futuras
-- Conversão da data para o padrao internacional
-- ==============================================================================
CREATE OR REPLACE TABLE lh_nautical.lh_nautical_db.slv_custos AS
WITH cte_historico_explodido AS (
    -- O EXPLODE cria uma nova linha para cada par {data, preco}
    SELECT 
        product_id,
        product_name,
        category AS original_category,
        EXPLODE(historic_data) AS price_record
    FROM lh_nautical.lh_nautical_db.custos_importacao
    WHERE product_id IS NOT NULL
)
-- Limpeza e Estruturação Final
SELECT 
    product_id,
    
    -- Padronização da Categoria
    CASE 
        WHEN UPPER(original_category) LIKE 'ELET%' THEN 'ELETRONICOS'
        WHEN UPPER(original_category) LIKE 'PROP%' THEN 'PROPULSAO'
        WHEN UPPER(original_category) LIKE '%NCOR%' THEN 'ANCORAGEM'
        ELSE UPPER(original_category)
    END AS category_clean,
    
    -- Extraindo o preço de dentro do struct
    price_record.usd_price AS unit_cost_usd,
    
    -- Limpeza da data e convertendo de string pra date
    TO_DATE(price_record.start_date, 'dd/MM/yyyy') AS start_date_clean
    
FROM cte_historico_explodido;

-- Validação
SELECT * FROM lh_nautical.lh_nautical_db.slv_custos LIMIT 15;

product_id,category_clean,unit_cost_usd,start_date_clean
1,ELETRONICOS,10583.63,2016-08-10
1,ELETRONICOS,8778.36,2018-06-15
1,ELETRONICOS,8023.87,2018-09-25
1,ELETRONICOS,8772.78,2019-03-19
1,ELETRONICOS,7918.18,2020-01-17
1,ELETRONICOS,6310.01,2020-06-17
1,ELETRONICOS,6586.7,2021-07-02
1,ELETRONICOS,6538.2,2022-05-16
1,ELETRONICOS,6360.91,2023-02-28
1,ELETRONICOS,6574.8,2023-10-17
